# 🛒 Blinkit Sales Dashboard (Interactive)

An interactive **Plotly Dash** dashboard built from the real Blinkit operational data:
`blinkit_orders.csv`, `blinkit_order_items.csv`, `blinkit_products.csv`, `blinkit_customers.csv`,
and `blinkit_customer_feedback.csv`.

**Layout:** yellow filter-panel sidebar + KPI cards + charts, in the same visual style as the
Blinkit Power BI reference dashboard.

**Filters (all live-update the whole page):** Category, Customer Segment, Payment Method,
Delivery Status, Order Date range.

**How to use:**
1. Make sure the CSVs above are in `/mnt/user-data/uploads/` (or edit `DATA_DIR` in the app cell below).
2. Run all cells top to bottom.
3. The last cell starts the Dash app **inline** in the notebook output — interact with the filters there.


In [ ]:
import subprocess, sys
# Uncomment if dash / dash-bootstrap-components aren't installed in your environment:
# subprocess.run([sys.executable, '-m', 'pip', 'install', 'dash', 'dash-bootstrap-components', '-q'])


## Build the interactive dashboard app

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output, dash_table
import dash_bootstrap_components as dbc

# ---------------------------------------------------------
# BRAND PALETTE
# ---------------------------------------------------------
YELLOW = '#F8CB45'
GREEN  = '#54B226'
DARK   = '#1F1F1F'
WHITE  = '#FFFFFF'
GREY   = '#5c5c5c'
CARD_BG = '#FFFFFF'
PAGE_BG = '#F4F4F2'
PALETTE = [YELLOW, GREEN, '#F2A900', '#3C8918', DARK, '#8BC34A', '#FFD873']

# ---------------------------------------------------------
# LOAD & MERGE DATA
# ---------------------------------------------------------
DATA_DIR = '/mnt/user-data/uploads/'

orders    = pd.read_csv(DATA_DIR + 'blinkit_orders.csv')
items     = pd.read_csv(DATA_DIR + 'blinkit_order_items.csv')
products  = pd.read_csv(DATA_DIR + 'blinkit_products.csv')
customers = pd.read_csv(DATA_DIR + 'blinkit_customers.csv')
feedback  = pd.read_csv(DATA_DIR + 'blinkit_customer_feedback.csv')

orders['order_date'] = pd.to_datetime(orders['order_date'])
orders['month'] = orders['order_date'].dt.to_period('M').dt.to_timestamp()
orders['weekday'] = orders['order_date'].dt.day_name()

df = (items
      .merge(products, on='product_id', how='left')
      .merge(orders[['order_id', 'order_date', 'month', 'weekday', 'payment_method',
                      'delivery_status', 'customer_id']], on='order_id', how='left')
      .merge(customers[['customer_id', 'area', 'customer_segment']], on='customer_id', how='left'))
df['revenue'] = df['quantity'] * df['unit_price']

order_rating = feedback.groupby('order_id')['rating'].mean().rename('order_rating')
orders_r = orders.merge(order_rating, on='order_id', how='left')

CATEGORIES = sorted(df['category'].dropna().unique())
SEGMENTS   = sorted(df['customer_segment'].dropna().unique())
PAYMENTS   = sorted(df['payment_method'].dropna().unique())
STATUSES   = sorted(df['delivery_status'].dropna().unique())
DATE_MIN, DATE_MAX = orders['order_date'].min(), orders['order_date'].max()

# ---------------------------------------------------------
# APP
# ---------------------------------------------------------
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP],
            title='Blinkit Sales Dashboard')
server = app.server

def kpi_card(title, value_id, icon):
    return dbc.Card(
        dbc.CardBody([
            html.Div(icon, style={'fontSize': '26px'}),
            html.H4(id=value_id, style={'fontWeight': '800', 'color': DARK, 'margin': '4px 0 0 0'}),
            html.Div(title, style={'fontSize': '12px', 'color': GREY, 'fontWeight': '600',
                                    'textTransform': 'uppercase', 'letterSpacing': '.5px'})
        ]),
        style={'backgroundColor': YELLOW, 'borderRadius': '14px', 'border': 'none',
               'boxShadow': '0 2px 8px rgba(0,0,0,0.08)'}
    )

sidebar = html.Div([
    html.Div([
        html.Span('blinkit', style={'color': DARK, 'fontWeight': '900', 'fontSize': '28px'}),
    ], style={'marginBottom': '2px'}),
    html.Div("India's Last Minute App", style={'color': DARK, 'fontSize': '11px',
                                                'fontWeight': '600', 'marginBottom': '30px'}),

    html.Div('FILTER PANEL', style={'color': DARK, 'fontWeight': '700', 'fontSize': '13px',
                                     'marginBottom': '14px', 'letterSpacing': '.5px'}),

    html.Label('Category', style={'color': DARK, 'fontSize': '12px', 'fontWeight': '600'}),
    dcc.Dropdown(id='f-category', options=[{'label': c, 'value': c} for c in CATEGORIES],
                 multi=True, placeholder='All', style={'marginBottom': '16px'}),

    html.Label('Customer Segment', style={'color': DARK, 'fontSize': '12px', 'fontWeight': '600'}),
    dcc.Dropdown(id='f-segment', options=[{'label': s, 'value': s} for s in SEGMENTS],
                 multi=True, placeholder='All', style={'marginBottom': '16px'}),

    html.Label('Payment Method', style={'color': DARK, 'fontSize': '12px', 'fontWeight': '600'}),
    dcc.Dropdown(id='f-payment', options=[{'label': p, 'value': p} for p in PAYMENTS],
                 multi=True, placeholder='All', style={'marginBottom': '16px'}),

    html.Label('Delivery Status', style={'color': DARK, 'fontSize': '12px', 'fontWeight': '600'}),
    dcc.Dropdown(id='f-status', options=[{'label': s, 'value': s} for s in STATUSES],
                 multi=True, placeholder='All', style={'marginBottom': '16px'}),

    html.Label('Order Date Range', style={'color': DARK, 'fontSize': '12px', 'fontWeight': '600'}),
    dcc.DatePickerRange(id='f-date', min_date_allowed=DATE_MIN, max_date_allowed=DATE_MAX,
                         start_date=DATE_MIN, end_date=DATE_MAX,
                         display_format='DD-MMM-YY', style={'marginBottom': '16px', 'fontSize': '11px'}),

    html.Button('Reset Filters', id='f-reset', n_clicks=0, style={
        'backgroundColor': DARK, 'color': WHITE, 'border': 'none', 'borderRadius': '8px',
        'padding': '8px 14px', 'fontWeight': '600', 'cursor': 'pointer', 'width': '100%'})
], style={'backgroundColor': YELLOW, 'padding': '24px 18px', 'height': '100%',
          'borderRadius': '18px'})

app.layout = html.Div([
    dbc.Row([
        dbc.Col(sidebar, width=2),
        dbc.Col([
            dbc.Row([
                dbc.Col(kpi_card('Total Sales', 'kpi-sales', '💰'), width=2),
                dbc.Col(kpi_card('Avg Order Value', 'kpi-avg', '📈'), width=2),
                dbc.Col(kpi_card('Total Orders', 'kpi-orders', '📦'), width=2),
                dbc.Col(kpi_card('Items Sold', 'kpi-items', '🧾'), width=2),
                dbc.Col(kpi_card('Avg Rating', 'kpi-rating', '⭐'), width=2),
                dbc.Col(kpi_card('On-Time %', 'kpi-ontime', '🚴'), width=2),
            ], className='g-2', style={'marginBottom': '14px'}),

            dbc.Row([
                dbc.Col(dcc.Graph(id='fig-trend', config={'displayModeBar': False}), width=8),
                dbc.Col(dcc.Graph(id='fig-payment', config={'displayModeBar': False}), width=4),
            ], className='g-2', style={'marginBottom': '14px'}),

            dbc.Row([
                dbc.Col(dcc.Graph(id='fig-category', config={'displayModeBar': False}), width=6),
                dbc.Col(dcc.Graph(id='fig-area', config={'displayModeBar': False}), width=6),
            ], className='g-2', style={'marginBottom': '14px'}),

            dbc.Row([
                dbc.Col(dcc.Graph(id='fig-status', config={'displayModeBar': False}), width=4),
                dbc.Col(dcc.Graph(id='fig-sentiment', config={'displayModeBar': False}), width=4),
                dbc.Col(dcc.Graph(id='fig-weekday', config={'displayModeBar': False}), width=4),
            ], className='g-2', style={'marginBottom': '14px'}),

            dbc.Row([
                dbc.Col(html.Div([
                    html.Div('SEGMENT PERFORMANCE', style={'fontWeight': '700', 'color': DARK,
                                                            'fontSize': '13px', 'marginBottom': '8px'}),
                    dash_table.DataTable(
                        id='tbl-segment',
                        style_header={'backgroundColor': DARK, 'color': WHITE, 'fontWeight': '700',
                                      'fontSize': '12px'},
                        style_cell={'textAlign': 'center', 'fontSize': '12px', 'padding': '8px',
                                    'backgroundColor': WHITE},
                        style_data_conditional=[
                            {'if': {'row_index': 'odd'}, 'backgroundColor': '#FBF3D9'}
                        ],
                    )
                ], style={'backgroundColor': CARD_BG, 'borderRadius': '14px', 'padding': '14px',
                          'boxShadow': '0 2px 8px rgba(0,0,0,0.06)'}), width=12),
            ], className='g-2'),

        ], width=10, style={'padding': '18px'})
    ])
], style={'backgroundColor': PAGE_BG, 'minHeight': '100vh', 'fontFamily': 'Segoe UI, Arial, sans-serif',
          'padding': '10px'})


def apply_filters(categories, segments, payments, statuses, start, end):
    d = df.copy()
    if categories:
        d = d[d['category'].isin(categories)]
    if segments:
        d = d[d['customer_segment'].isin(segments)]
    if payments:
        d = d[d['payment_method'].isin(payments)]
    if statuses:
        d = d[d['delivery_status'].isin(statuses)]
    if start and end:
        d = d[(d['order_date'] >= start) & (d['order_date'] <= end)]
    return d


@app.callback(
    Output('kpi-sales', 'children'), Output('kpi-avg', 'children'),
    Output('kpi-orders', 'children'), Output('kpi-items', 'children'),
    Output('kpi-rating', 'children'), Output('kpi-ontime', 'children'),
    Output('fig-trend', 'figure'), Output('fig-payment', 'figure'),
    Output('fig-category', 'figure'), Output('fig-area', 'figure'),
    Output('fig-status', 'figure'), Output('fig-sentiment', 'figure'),
    Output('fig-weekday', 'figure'),
    Output('tbl-segment', 'data'), Output('tbl-segment', 'columns'),
    Input('f-category', 'value'), Input('f-segment', 'value'),
    Input('f-payment', 'value'), Input('f-status', 'value'),
    Input('f-date', 'start_date'), Input('f-date', 'end_date'),
)
def update_dashboard(categories, segments, payments, statuses, start, end):
    d = apply_filters(categories, segments, payments, statuses, start, end)
    order_ids = d['order_id'].unique()
    o = orders_r[orders_r['order_id'].isin(order_ids)]
    fb = feedback[feedback['order_id'].isin(order_ids)]

    total_sales = d['revenue'].sum()
    avg_order = o['order_total'].mean() if len(o) else 0
    total_orders = o['order_id'].nunique()
    total_items = d['quantity'].sum()
    avg_rating = fb['rating'].mean() if len(fb) else 0
    ontime_pct = (o['delivery_status'] == 'On Time').mean() * 100 if len(o) else 0

    def money(v):
        if v >= 1e7: return f"₹{v/1e7:.2f}Cr"
        if v >= 1e5: return f"₹{v/1e5:.2f}L"
        if v >= 1e3: return f"₹{v/1e3:.1f}K"
        return f"₹{v:.0f}"

    def style_ax(fig, title):
        fig.update_layout(
            title=dict(text=title, font=dict(size=14, color=DARK, family='Segoe UI', weight='bold')),
            paper_bgcolor=CARD_BG, plot_bgcolor=CARD_BG,
            font=dict(color=DARK, family='Segoe UI'),
            margin=dict(l=40, r=20, t=44, b=30), height=300,
        )
        return fig

    # Trend
    trend = o.groupby(o['month'])['order_total'].sum().sort_index() if len(o) else pd.Series(dtype=float)
    fig_trend = go.Figure()
    fig_trend.add_trace(go.Scatter(x=trend.index, y=trend.values, mode='lines+markers',
                                    line=dict(color=GREEN, width=3), fill='tozeroy',
                                    fillcolor='rgba(248,203,69,0.45)'))
    style_ax(fig_trend, 'Monthly Sales Trend')

    # Payment donut
    pay = d.groupby('payment_method')['revenue'].sum()
    fig_payment = go.Figure(go.Pie(labels=pay.index, values=pay.values, hole=0.55,
                                    marker=dict(colors=PALETTE)))
    style_ax(fig_payment, 'Sales by Payment Method')

    # Category bar
    cat = d.groupby('category')['revenue'].sum().sort_values().tail(10)
    fig_category = go.Figure(go.Bar(x=cat.values, y=cat.index, orientation='h',
                                     marker=dict(color=YELLOW, line=dict(color=DARK, width=1))))
    style_ax(fig_category, 'Sales by Category')

    # Area bar
    area = d.groupby('area')['revenue'].sum().sort_values(ascending=False).head(10).sort_values()
    fig_area = go.Figure(go.Bar(x=area.values, y=area.index, orientation='h',
                                 marker=dict(color=GREEN, line=dict(color=DARK, width=1))))
    style_ax(fig_area, 'Top 10 Areas by Sales')

    # Delivery status donut
    stat = o['delivery_status'].value_counts()
    fig_status = go.Figure(go.Pie(labels=stat.index, values=stat.values, hole=0.55,
                                   marker=dict(colors=[GREEN, YELLOW, '#C0392B'])))
    style_ax(fig_status, 'Delivery Status')

    # Sentiment donut
    sent = fb['sentiment'].value_counts()
    fig_sentiment = go.Figure(go.Pie(labels=sent.index, values=sent.values, hole=0.55,
                                      marker=dict(colors=[GREEN, '#F2A900', '#C0392B'])))
    style_ax(fig_sentiment, 'Feedback Sentiment')

    # Weekday bar
    weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    wk = o['weekday'].value_counts().reindex(weekday_order).fillna(0)
    fig_weekday = go.Figure(go.Bar(x=wk.index, y=wk.values,
                                    marker=dict(color=DARK)))
    style_ax(fig_weekday, 'Orders by Weekday')

    # Segment table
    seg_tbl = d.groupby('customer_segment').agg(
        Total_Sales=('revenue', 'sum'),
        No_of_Items=('quantity', 'sum'),
    ).reset_index()
    seg_orders = o.groupby(o['customer_id'].map(customers.set_index('customer_id')['customer_segment']))['order_total'].agg(['mean', 'count'])
    seg_tbl['Total_Sales'] = seg_tbl['Total_Sales'].map(money)
    seg_tbl = seg_tbl.rename(columns={'customer_segment': 'Segment', 'No_of_Items': 'Items Sold'})
    avg_rating_seg = fb.merge(customers[['customer_id', 'customer_segment']], on='customer_id', how='left') \
                       .groupby('customer_segment')['rating'].mean().round(2)
    seg_tbl['Avg Rating'] = seg_tbl['Segment'].map(avg_rating_seg)
    columns = [{'name': c, 'id': c} for c in seg_tbl.columns]
    data = seg_tbl.to_dict('records')

    return (money(total_sales), money(avg_order), f"{total_orders:,}", f"{int(total_items):,}",
            f"{avg_rating:.2f}" if avg_rating == avg_rating else "N/A", f"{ontime_pct:.0f}%",
            fig_trend, fig_payment, fig_category, fig_area, fig_status, fig_sentiment, fig_weekday,
            data, columns)


@app.callback(
    Output('f-category', 'value'), Output('f-segment', 'value'),
    Output('f-payment', 'value'), Output('f-status', 'value'),
    Output('f-date', 'start_date'), Output('f-date', 'end_date'),
    Input('f-reset', 'n_clicks'), prevent_initial_call=True
)
def reset_filters(n):
    return None, None, None, None, DATE_MIN, DATE_MAX


## Run the dashboard
Running this cell launches the dashboard **inline, right below the cell**. Use the filter panel to slice the data — every chart, KPI and table updates live.

In [ ]:
# Runs inline inside Jupyter (JupyterLab / Notebook / VS Code).
# If you're in a plain Jupyter Notebook and 'inline' mode doesn't render, change jupyter_mode to 'external'
# and open the printed http://127.0.0.1:8050 link in your browser instead.
app.run(debug=False, port=8050, jupyter_mode='inline', jupyter_height=1000)
